In [5]:
import pandas as pd
import sys,os,inspect
import calendar
from datetime import datetime
from datetime import timedelta
#from cronsim import CronSim
import argparse
import json,time

import ast
#import pygsheets
#from tabulate import tabulate
#from difflib import SequenceMatcher
import shutil

#import PySimpleGUI as sg
#import PySimpleGUIWeb as sg
import pathlib
#from wordcloud import WordCloud, STOPWORDS
#import matplotlib.pyplot as plt
#import textwrap
import re
from shlex import split
import webbrowser
import subprocess
#import collections
#import pyglet,tkinter
#from pyglet import font
#from tkinter import Tk
#import pygments.lexers
#from chlorophyll import CodeView
#import tkinter as tk
#from tkinter import ttk,Button,Label


#font.add_file('/etc/fonts/fonts/CENTAUR.TTF')
#font='Courier 10 bold '
bicHome = "/home/joe/bic_etl/"
numWindows=0
headerStore = {}

colorPairs = [["#D6EAF8","#85C1E9"],["#b3f0ff","#33d6ff"],["#D5F5E3","#A3E4D7"],["#FCF3CF","#F7DC6F"]]
windowsOpen = {}
windowsOpen["main"] = []
windowsOpen["unique"] = []


import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials

global datasets

if sys.platform == "linux":
    path="/home/joe/work/Logs/logs"
else:
    path= "\\\\wsl.localhost\\Ubuntu\\home\\joe\\work\\Logs\\"

logging = 2
today = datetime.today()
## Directory where dataset difintions will be store... this will be inside
## bic_etl/dataset_dirs/definitionDir
definitionDir = "defs" 
sourceDefFile = " "
nfieldsAdded = 0  # numbers of XREF Fields added
nfieldsAddedS = 0  # numbers of XREF Fields added for from scratch


## map etl dataset names to invenotry  dataset names:
mapTitles = {}
#mapTitles[etl title] = invenotry title
mapTitles["Other Names a Registered Entity Uses to Solicit Contributions"] = \
"Other Names a Registered Entity Uses to Solicit Contributions in Colorado"

mapTitles["Registration for Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado"] =  \
"Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado"



In [6]:
def getXrefs():
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive.file",
                  "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name('/home/joe/work/client_secret.json',
     scope)
    client = gspread.authorize(creds)

    gc = gspread.service_account("/home/joe/work/client_secret.json")
    # for gg in gc.list_spreadsheet_files():
    #      print("GGGGG ",gg)
    # https://docs.google.com/spreadsheets/d/1WTaOglzbSsYiHhAGguGxHQXmAGmOhfFHkGkMLowxAOA/edit?usp=sharing
    sheet = client.open('BIC Data Inventory and Metadata').worksheet(
        'Maintenance_Framework')
    repo_sheet = client.open('BIC Data Inventory and Metadata').worksheet(
        'MetadataRepository')
    fields_sheet = client.open('BIC Data Inventory and Metadata').worksheet(
        'Field Descriptions')

    dfRepo = pd.DataFrame(repo_sheet.get_all_records(head=3))
    xrefsBy4x4 = {}
    xrefsByTitle = {}

    for index,row in dfRepo[['Dataset Title','Socrata Link']].iterrows():
        xrefsByTitle[row['Dataset Title']] = row['Socrata Link']
        xrefsBy4x4[row['Socrata Link']] = row['Dataset Title']
        
    dfFields = pd.DataFrame(fields_sheet.get_all_records(head=1))
    fields = {}
    for index,row in dfFields.iterrows():
        s4x4 = row["Socrata ID"]
        of = row["Source Field Name"]
        tf = row["Full Field Name"]
        af = row["API Field Name"]
        des = row["Description"]
        if s4x4 in fields:
            fields[s4x4]["source"].append(of)
            fields[s4x4]["cim"].append(tf)
            fields[s4x4]["api"].append(af)
            fields[s4x4]["description"].append(des)
        
        else:
            fields[s4x4] = {}
            fields[s4x4]["source"] = []
            fields[s4x4]["cim"] = []
            fields[s4x4]["api"] = []
            fields[s4x4]["description"] = []
            
            
            fields[s4x4]["source"].append(of)
            fields[s4x4]["cim"].append(tf)
            fields[s4x4]["api"].append(af)
            fields[s4x4]["description"].append(des)
            
        
        
        
    return xrefsBy4x4,xrefsByTitle,fields


In [7]:
xrefsBy4x4,xrefsByTitle,fields = getXrefs()

In [ ]:
def splitL(data):
    if data:
        head, *tail = data  # This is a nicer way of doing head, tail = data[0], data[1:]
        return {head: splitL(tail)}
    else:
        return []

def go_deeper(aDict,value,nit):
    nit+=1
    for k, v in aDict.items():
         if not bool(v):            
             aDict[k] = []
             aDict[k].append(value)
         elif isinstance(v,list):
             aDict[k].append(value)
         else:
             go_deeper(v,value,nit)
   
    return aDict



def init():
    global dataSetsEtl,datasets,groupMenu
    groups = []
    desktop = pathlib.Path("/home/joe/bic_etl")
    runEtls = []
    dataSets = []
    info = {}
    # .rglob() produces a generator too
    desktop.rglob("*")
    files = list(desktop.rglob("*"))
# Which you can wrap in a list() constructor to materialize
    for ff in files:
            print(ff)

            if (str(ff).split("/")[-1] == "run_etl.json"):     
                a=str(ff).split("/")
                m = a.index("bic_etl")
                b = a[m+1:-1]
                ll = splitL(b)
                groups.append(ll)
                runEtls.append(ff)
            
 #   print(f"{len(runEtls)} run_etl.json files found")    
    
    dataSets = []
    groups = []
    for file in runEtls:
      f = open(file,"r")
      data = json.load(f)
      # print(file)
      # print("----")
      a=str(file).split("/")
      m = a.index("bic_etl")
      b = a[m+1:-1]
      group = b[0]
      ll = splitL(b)
      bdir = "/".join(b)  
    
      
    #  groups.append(ll)
      for val in data:
            if "title" in val:
                  title=val["title"]
                  if title in mapTitles:
                     title = mapTitles[title]
                  info[title]={}
                  info[title]["directory"] = bdir
                  info[title]["group"] = group
                
    #              print(title,ll)
                  nit=0
                  ll = go_deeper(ll,title,nit)
             #     info[title]["groups"] = ll
            
     #             print(ll)
     #             groups.append(ll)
            dataSets.append(val)
    #  print("FF ",ll) 
      groups.append(ll)

    datasets = []
    dataSetsEtl={}
    for val in dataSets:
        if "title" in val:
          datasets.append(val["title"])
          title=val["title"]
          if title in mapTitles:
            title = mapTitles[title]
          dataSetsEtl[title] = {}  
          dataSetsEtl[title]["info"] = {}
          dataSetsEtl[title]["info"]["directory"] = info[title]["directory"]
          dataSetsEtl[title]["info"]["group"] = info[title]["group"]
     #     dataSetsEtl[title]["info"]["groups"] = info[title]["groups"]
            
        
            
          for k,v in val.items():
          #      print(k,v)
                if k != "title":
                    dataSetsEtl[title][k] = {}
                    if isinstance(v,dict):
                        for k1,v1 in v.items():
                            dataSetsEtl[title][k][k1]=v1
                            
    return sorted(datasets)

                            
datasets = init()

In [22]:
for title in dataSetsEtl.keys():
    if 'extract' in dataSetsEtl[title]:
        print(title,dataSetsEtl[title]['extract'])

CIM Catalog Download {'language': 'node', 'file': 'general/scripts/request_url.js', 'options': ['-u https://data.colorado.gov/api/views/e7nm-tn2z/rows.csv?accessType=DOWNLOAD', '-f catalog/data_source/inventory-$(date +%F).csv']}
Durable Medical Equipment Suppliers in Colorado {'language': 'node', 'file': 'general/scripts/sftp_extract.js', 'options': ['-f DME/CurrentDMESuppliers-CIM.txt', '-o cdos/health/data_source/', '-a .tsv']}
Current Notaries in Colorado {'language': 'node', 'file': 'general/scripts/sftp_extract.js', 'options': ['-f notary/CurrentCommissionedNotaries.txt', '-o cdos/government/data_source/', '-a .tsv']}
Uniform Commercial Code (UCC) Collateral Information in Colorado {}
Uniform Commercial Code (UCC) Debtor Information in Colorado {'language': 'node', 'file': 'general/scripts/sftp_extract.js', 'options': ['-f uccbulkdata/full/uccdbtr.txt', '-o cdos/business/ucc/data_source/', '-a .tsv']}
Uniform Commercial Code (UCC) Filing Information in Colorado {'language': 'node